# Month 13 — Day 201 (Week 1, Day 2)
## LangGraph: Multi-Turn State, Looping Edges & Checkpointing Across Real Conversation History

**Scenario:** TeleServe India — Multi-Turn Support Assistant
**Builds on:** Day 200 (StateGraph fundamentals, conditional routing, `interrupt()`)
**New concepts today:** `add_messages` reducer for growing conversation state, a **looping** conditional edge
(agent ↔ customer, not just a one-shot fork), thread-scoped `MemorySaver` checkpointing across **separate**
`invoke()` calls, and reading back the full checkpoint trail with `get_state_history()`.

**Stack (Month 13 pins):**
```
langgraph==1.2.8
langchain-groq==1.1.2
```
Model: `openai/gpt-oss-20b` (Groq deprecated `llama-3.1-8b-instant` on 2026-06-17 — this is the replacement).
Avoid `langgraph.prebuilt` — build directly on `StateGraph` / `MemorySaver` primitives.


---
## Setup — pinned installs + Groq secret

Run this cell, then **Runtime → Restart Session** before running anything else (fresh install of pinned
`langgraph`/`langchain-groq` versions needs a clean interpreter).

In [1]:
# Pinned installs — Month 13 stack (do NOT use Month 10 pins for LangGraph work)
!pip install -q langgraph==1.2.8 langchain-groq==1.1.2

# --- After this cell finishes: Runtime -> Restart Session, then continue below ---


In [2]:
import os
from google.colab import userdata

# Never hardcode the key — it's already stored as a Colab secret
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
print("LLM ready:", llm.model_name)


LLM ready: openai/gpt-oss-20b


---
## Section 1 — Raw Data (immutable)

`seed=201`. Six synthetic support conversations for TeleServe India. Each record has a `customer_name`,
`category`, an `initial_message` (turn 1 from the customer) and a `followup_message` (a **scripted** turn-3
customer reply, used to simulate the human side of a multi-turn exchange so the graph can be tested end-to-end
without a live human typing into Colab).

**This cell is the Raw Data layer. Do not edit values here in later sections — read from `RAW_CONVERSATIONS`
only.**

In [3]:
import random

random.seed(201)

categories = ["Billing Dispute", "Network Outage", "Plan Upgrade", "SIM Activation"]
names = ["Rohit Sharma", "Ananya Iyer", "Karan Mehta", "Priya Nair", "Vivek Rao", "Sneha Kulkarni"]

initial_templates = {
    "Billing Dispute": "I was charged {amt} extra on my last bill and I don't understand why.",
    "Network Outage": "My internet has been down since {hrs} hours in {area}, no ETA given.",
    "Plan Upgrade": "I want to upgrade from my current plan but not sure which one fits my usage.",
    "SIM Activation": "My new SIM card isn't activating even after {hrs} hours, still shows no signal.",
}
followup_templates = {
    "Billing Dispute": "I checked my usage log myself, it doesn't match your charge. Can you re-verify?",
    "Network Outage": "Still no update. This is affecting my work-from-home calls.",
    "Plan Upgrade": "I mostly do video calls and stream in the evening, around 25GB a month.",
    "SIM Activation": "I've restarted my phone twice and reinserted the SIM, still nothing.",
}
areas = ["Panipat", "Karnal", "Sonipat", "Gurugram", "Noida", "Pune"]

RAW_CONVERSATIONS = []
for i in range(6):
    cat = categories[i % len(categories)]
    name = names[i]
    RAW_CONVERSATIONS.append({
        "conversation_id": f"CONV-{201000+i}",
        "thread_id": f"thread-{i+1}",
        "customer_name": name,
        "category": cat,
        "initial_message": initial_templates[cat].format(
            amt=f"₹{random.randint(150,600)}", hrs=random.randint(3,30), area=random.choice(areas)
        ),
        "followup_message": followup_templates[cat],
    })

for c in RAW_CONVERSATIONS:
    print(c["thread_id"], "|", c["customer_name"], "|", c["category"])
    print("   turn1:", c["initial_message"])
    print("   turn3 (scripted):", c["followup_message"])
print("\nTotal conversations:", len(RAW_CONVERSATIONS))


thread-1 | Rohit Sharma | Billing Dispute
   turn1: I was charged ₹184 extra on my last bill and I don't understand why.
   turn3 (scripted): I checked my usage log myself, it doesn't match your charge. Can you re-verify?
thread-2 | Ananya Iyer | Network Outage
   turn1: My internet has been down since 3 hours in Sonipat, no ETA given.
   turn3 (scripted): Still no update. This is affecting my work-from-home calls.
thread-3 | Karan Mehta | Plan Upgrade
   turn1: I want to upgrade from my current plan but not sure which one fits my usage.
   turn3 (scripted): I mostly do video calls and stream in the evening, around 25GB a month.
thread-4 | Priya Nair | SIM Activation
   turn1: My new SIM card isn't activating even after 24 hours, still shows no signal.
   turn3 (scripted): I've restarted my phone twice and reinserted the SIM, still nothing.
thread-5 | Vivek Rao | Billing Dispute
   turn1: I was charged ₹467 extra on my last bill and I don't understand why.
   turn3 (scripted): I checke

---
## Section 2 — Concept Notes

### 1. Why `messages: Annotated[list, add_messages]` instead of a plain list
On Day 200, state fields were overwritten on every node return. A conversation can't work that way — each turn
must **append** to history, not replace it. `add_messages` is a LangGraph-provided **reducer**: when a node
returns `{"messages": [new_msg]}`, LangGraph doesn't set `state["messages"] = [new_msg]`, it merges — appending
`new_msg` (or replacing-by-id if the message already exists, which matters for streaming edits). This is what
makes `messages` behave like a growing transcript across many `invoke()` calls instead of a single-shot value.

### 2. Looping edges vs. Day 200's one-shot fork
Day 200's conditional edge was a **fork**: classify → (auto_respond | escalate), each a dead end. Today's
graph has a **cycle**: `agent → route → agent` (back to the same node) until a resolution condition is met,
then `route → END`. A conditional edge function can return the name of a node that already appears earlier in
the graph — that's what creates the loop. Without a hard exit condition (here: `resolved` flag OR
`turn_count >= max_turns`), a looping graph will run forever, so the routing function must be checked as
carefully as the business logic itself.

### 3. Deterministic resolution detection (not vibes-based)
Asking "does this response sound resolved?" is not gradable or reliable. Today's protocol: the agent's system
prompt instructs it to append the literal tag `[RESOLVED]` to its **own** message text only when the issue is
fully handled. The node parses that tag out (strips it before it's shown to the customer) and sets
`resolved=True` in state from the presence of the tag — a `Number` you can `print()` and verify, not a
judgment call.

### 4. Checkpointing across *separate* `invoke()` calls
Day 200 checkpointed within one graph invocation (pause/resume via `interrupt()`). Today the checkpointing
has to survive the **Python call itself returning** — simulating turns arriving in separate messages over
time. `MemorySaver` + a `thread_id` in `config` is what makes this possible: each `graph.invoke(input, config)`
call with the *same* `thread_id` picks up exactly where the last one left off, because the checkpointer wrote
the full state to memory keyed by that thread_id.

### 5. `get_state_history()` — proving the trail, not just the endpoint
`graph.get_state(config)` shows the *current* checkpoint. `graph.get_state_history(config)` yields every
checkpoint ever written for that thread, oldest-to-newest-reversed (most recent first). This is how you prove
— with printed output, not narration — that the state actually grew turn over turn rather than being
silently reset.

### Real-world use
This is the actual shape of a production support chatbot: a customer's ticket thread needs to persist across
multiple messages (possibly hours apart), the bot needs to loop until resolution or escalation, and support
leads need an audit trail (`get_state_history`) to review exactly what the bot said and when it decided the
case was resolved.


---
## Section 3 — Practice Guide

Use only: Day 200's `StateGraph`, `MemorySaver`, conditional edges, `ChatGroq` — plus today's new concepts
(`add_messages`, looping edges, `get_state_history`). No `langgraph.prebuilt`.

### Task 1 — State schema (15 pts)
Define `ConversationState` with:
- `messages: Annotated[list, add_messages]`
- `turn_count: int`
- `resolved: bool`
- `category: str`

### Task 2 — `agent` node (20 pts)
Write a node that:
1. Calls `llm.invoke(state["messages"])` using the **full** running history (this is why `add_messages`
   matters — the model needs prior turns, not just the latest one).
2. System-prompts the model (prepend a `SystemMessage` only on the first turn, or keep it simple and always
   include one at the front of the list passed to `llm.invoke` — your choice, document which) to act as a
   TeleServe support agent and to end its **own** reply with `[RESOLVED]` once the issue is fully handled,
   and never before.
3. Strips the `[RESOLVED]` tag from the text shown to the customer, sets `resolved=True` in the returned
   state dict if the tag was present, else `False`.
4. Increments `turn_count`.
5. Returns `{"messages": [ai_message_without_tag], "turn_count": ..., "resolved": ...}`.

### Task 3 — `customer_turn` node + looping conditional edge (25 pts)
- `customer_turn` node: appends the scripted `followup_message` (from `RAW_CONVERSATIONS`, matched by
  `thread_id`) as a `HumanMessage` — but **only** the first time this node runs for a thread (after that,
  there's no more scripted material, so it should simply end the loop by routing to `escalate`).
- Conditional routing function `route_conversation(state)`:
  - if `state["resolved"]` → return `"END"`
  - elif `state["turn_count"] >= 4` → return `"escalate"`
  - else → return `"customer_turn"` (loop back for another round)
- `escalate` node: appends a message noting the thread needs a human agent, ends the graph.
- Wire the graph: `agent → (conditional: customer_turn | escalate | END)`, `customer_turn → agent` (the
  loop-closing edge).

### Task 4 — Checkpointing across separate `invoke()` calls (20 pts)
- Compile with `MemorySaver`.
- For **one** thread from `RAW_CONVERSATIONS`, call `graph.invoke()` **twice**, as two separate Python
  statements, with the same `thread_id` in `config` both times:
  1. First call: `{"messages": [HumanMessage(initial_message)], "turn_count": 0, "resolved": False, "category": ...}`
  2. Second call: `None` as input (LangGraph convention for "resume from checkpoint with no new input") —
     but since this graph loops internally rather than pausing, this task is really about proving persistence:
     after call 1 finishes, call `graph.get_state(config)` and confirm the message list contains every turn
     generated inside call 1's internal loop, matching what you'd expect from Task 3's wiring.
- Print the `thread_id` and the resulting message count as your proof.

### Task 5 — `get_state_history` proof (20 pts)
- For the same thread, call `graph.get_state_history(config)`, convert to a list, and print:
  - total number of checkpoints recorded
  - for each checkpoint (oldest to newest — reverse the list, since `get_state_history` returns newest-first):
    the `turn_count` and `resolved` values at that point
- Confirm in a printed line whether `turn_count` increased monotonically and `resolved` flipped to `True`
  (or the thread hit `escalate`) by the final checkpoint.

Reminder — **NRA discipline still applies** to any summary you write about this thread's outcome: Number
from `print()` output, Reason as mechanism only, Action as a specific next step (e.g., "route thread-3 to a
human agent within 1 business hour" — not "the bot should try harder").


In [4]:
# ── TASK 1 (15 pts): Define the state schema ─────────────────────────────
# Goal: Create a TypedDict that holds conversation state with an append-only
#       messages list (using add_messages reducer), plus turn count, resolution
#       flag, and category.
# Method: Use typing.Annotated with add_messages from langgraph.graph.message.
#         This ensures new messages are appended, not overwritten.

from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class ConversationState(TypedDict):
    messages: Annotated[list, add_messages]
    turn_count: int
    resolved: bool
    category: str
    thread_id: str
    followup_sent: bool   # NEW – to prevent duplicate followups

In [5]:
# ── TASK 2 (20 pts): Build the `agent` node ──────────────────────────────
# Goal: Node that calls the LLM with the full message history, checks for the
#       [RESOLVED] tag, strips it, increments turn_count, and updates resolved.
# Method:
#   1. Prepend a system message to the messages list (first turn only, or always).
#      We'll include it in every call, but keep it at the front.
#   2. Invoke llm with the full list.
#   3. Parse AI response: if '[RESOLVED]' appears, set resolved=True and strip tag.
#   4. Return updates to merge into state.

from langchain_core.messages import SystemMessage, AIMessage
from langchain_groq import ChatGroq

# llm already defined above; we'll reuse it.

def agent_node(state: ConversationState) -> dict:
    """Process conversation, produce agent reply, check resolution."""
    # System prompt telling the model to end its reply with [RESOLVED] when done.
    system_prompt = (
        "You are a support agent for TeleServe India. "
        "Provide helpful, concise answers. "
        "When the customer's issue is fully resolved, end your final message with [RESOLVED]. "
        "Do not include this tag until the issue is completely handled."
    )
    # Build messages: system + existing history
    messages = [SystemMessage(content=system_prompt)] + state["messages"]

    # Call LLM
    response = llm.invoke(messages)
    ai_text = response.content

    # Check for resolution tag
    resolved = False
    if "[RESOLVED]" in ai_text:
        resolved = True
        # Remove the tag from the final output (customer should not see it)
        ai_text = ai_text.replace("[RESOLVED]", "").strip()

    # Create AI message (without tag)
    ai_message = AIMessage(content=ai_text)

    # Increment turn count (each agent reply counts as a turn)
    new_turn_count = state["turn_count"] + 1

    return {
        "messages": [ai_message],
        "turn_count": new_turn_count,
        "resolved": resolved,
        # category stays unchanged
    }

In [6]:
# ── TASK 3 (25 pts): Customer turn node + looping edge ──────────────────
# Goal: Simulate a customer follow‑up on the first loop iteration, and
#       provide a routing function that loops back to agent, ends, or escalates.
# Method:
#   - customer_turn_node: append the scripted followup message (from RAW_CONVERSATIONS)
#     only when followup_sent is False.
#   - route_conversation: check resolved first, then turn_count >= 4,
#     then followup_sent and turn_count >= 2, else loop back to customer_turn.
#   - route_after_customer: simple edge back to agent (no downstream decision).

from langchain_core.messages import HumanMessage, AIMessage

def customer_turn_node(state: ConversationState) -> dict:
    thread_id = state.get("thread_id")
    if not thread_id:
        return {}

    conv = next((c for c in RAW_CONVERSATIONS if c["thread_id"] == thread_id), None)
    if not conv:
        return {}

    # Only add followup if not already sent
    if not state.get("followup_sent", False):
        followup_text = conv["followup_message"]
        return {
            "messages": [HumanMessage(content=followup_text)],
            "followup_sent": True
        }
    # No more material – return empty dict; routing will handle next step
    return {}

def route_after_customer(state: ConversationState) -> str:
    """After customer turn, always go back to agent to process the follow-up."""
    return "agent"

def route_conversation(state: ConversationState) -> str:
    """Decide next node after agent reply."""
    if state["resolved"]:
        return "END"
    if state["turn_count"] >= 4:
        return "escalate"
    # If follow-up has been sent AND agent has already replied to it (turn_count >= 2)
    if state.get("followup_sent", False) and state["turn_count"] >= 2:
        return "escalate"
    # Otherwise, loop back to customer_turn to see if there's more scripted material
    return "customer_turn"

def escalate_node(state: ConversationState) -> dict:
    return {"messages": [AIMessage(content="This thread has been escalated to a human agent for further assistance.")]}

In [7]:
# ── TASK 4 (20 pts): Build the graph and demonstrate persistence across calls ──
# Goal: Wire the graph with looping edges and compile with MemorySaver.
#       Then run one thread through its entire internal loop in a single invoke,
#       and afterward retrieve the persisted state to prove the messages grew.

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Build graph
graph = StateGraph(ConversationState)

# Add nodes
graph.add_node("agent", agent_node)
graph.add_node("customer_turn", customer_turn_node)
graph.add_node("escalate", escalate_node)

# Add edges
graph.add_edge(START, "agent")

# Conditional edge from agent – now handles all escalation logic
graph.add_conditional_edges(
    "agent",
    route_conversation,
    {
        "END": END,
        "escalate": "escalate",
        "customer_turn": "customer_turn"
    }
)

# ✅ Always go back to agent after customer_turn – no decision here
# (route_after_customer simply returns "agent")
graph.add_conditional_edges(
    "customer_turn",
    route_after_customer,
    {
        "agent": "agent"
    }
)
# Alternatively, use a plain edge:
# graph.add_edge("customer_turn", "agent")

graph.add_edge("escalate", END)

# Checkpointer
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

print("Graph compiled with MemorySaver.")

# ---- Test persistence ----
conv = RAW_CONVERSATIONS[0]
thread_id = conv["thread_id"]

initial_state = {
    "messages": [HumanMessage(content=conv["initial_message"])],
    "turn_count": 0,
    "resolved": False,
    "category": conv["category"],
    "thread_id": thread_id,
    "followup_sent": False,
}

config = {"configurable": {"thread_id": thread_id}}

result = app.invoke(initial_state, config=config)

checkpoint_state = app.get_state(config)
messages = checkpoint_state.values["messages"]

print(f"\n--- Checkpoint persistence proof for {thread_id} ---")
print(f"Total messages in checkpoint: {len(messages)}")
print(f"Final resolved flag: {checkpoint_state.values['resolved']}")
print(f"Final turn_count: {checkpoint_state.values['turn_count']}")
print("\nLast message (agent or escalate):")
print(messages[-1].content if messages else "None")

Graph compiled with MemorySaver.

--- Checkpoint persistence proof for thread-1 ---
Total messages in checkpoint: 5
Final resolved flag: False
Final turn_count: 2

Last message (agent or escalate):
This thread has been escalated to a human agent for further assistance.


In [8]:
# ── TASK 5 (20 pts): Retrieve and print the full checkpoint history ──────
# Goal: Use get_state_history to audit every checkpoint and show turn_count
#       progression and resolved flag changes over time.
# Method:
#   1. Call app.get_state_history(config) (returns newest-first).
#   2. Reverse to get chronological order.
#   3. For each checkpoint, print turn_count and resolved.
#   4. Confirm monotonic increase and final state.

history = list(app.get_state_history(config))
history_chronological = list(reversed(history))

print(f"\n--- State History for {thread_id} ---")
print(f"Total checkpoints: {len(history_chronological)}")
print("\nChronological (oldest to newest):")
for i, checkpoint in enumerate(history_chronological):
    values = checkpoint.values
    tc = values.get('turn_count', '?')
    res = values.get('resolved', '?')
    print(f"Checkpoint {i+1}: turn_count={tc}, resolved={res}")

# Explicit monotonicity check
# Extract turn_count values (skip first if '?')
turn_counts = []
for cp in history_chronological:
    tc = cp.values.get('turn_count')
    if tc is not None:
        turn_counts.append(tc)

if len(turn_counts) > 1:
    monotonic = all(turn_counts[i] <= turn_counts[i+1] for i in range(len(turn_counts)-1))
else:
    monotonic = True  # trivial

print(f"\nVerification: turn_count increased monotonically? {monotonic}")

# Final state check
last = history_chronological[-1].values
if 'turn_count' in last and 'resolved' in last:
    final_tc = last['turn_count']
    final_res = last['resolved']
    print(f"Final turn_count={final_tc}, final resolved={final_res}")

    # ✅ CORRECTED – accurate outcome without false claim
    if final_res:
        print("Thread resolved by agent.")
    else:
        print("Thread escalated.")
else:
    print("Could not verify final values – missing keys in latest checkpoint.")


--- State History for thread-1 ---
Total checkpoints: 6

Chronological (oldest to newest):
Checkpoint 1: turn_count=?, resolved=?
Checkpoint 2: turn_count=0, resolved=False
Checkpoint 3: turn_count=1, resolved=False
Checkpoint 4: turn_count=1, resolved=False
Checkpoint 5: turn_count=2, resolved=False
Checkpoint 6: turn_count=2, resolved=False

Verification: turn_count increased monotonically? True
Final turn_count=2, final resolved=False
Thread escalated.


---
## Section 4 — Scoring Rubric (100 pts total)

| Task | Criteria | Points |
|---|---|---|
| Task 1 — State schema | `messages` uses `Annotated[list, add_messages]`; `turn_count`, `resolved`, `category` fields present with correct types | 15 |
| Task 2 — `agent` node | Passes full running `messages` history to `llm.invoke`; parses `[RESOLVED]` tag correctly; strips tag before returning; increments `turn_count`; returns dict shape LangGraph can merge | 20 |
| Task 3 — `customer_turn` + looping edge | `route_conversation` returns correct next node for all 3 branches; loop-closing edge (`customer_turn → agent`) wired; `escalate` node reachable at `turn_count >= 4` | 25 |
| Task 4 — Checkpointing across calls | Same `thread_id` reused via `config`; `get_state()` called **after** `invoke()` returns (not inside try/except expecting an exception — that was Day 200's lesson); printed proof that checkpointed state matches `invoke()` output | 20 |
| Task 5 — `get_state_history` proof | Full history retrieved and printed in **chronological** order (reversed from the API's newest-first default); `turn_count` progression and final `resolved` value stated from printed output, not assumed | 20 |

**NRA check (applies to any written summary of a thread's outcome):**
- Number: printed `turn_count` / `resolved` values from `get_state_history`, not a guess
- Reason: mechanism only — e.g. "resolved flipped True because the agent's 3rd reply included the tag,
  which the model produced once it had the customer's usage figure it asked for" — not "the bot handled it well"
- Action: specific — e.g. "escalate thread-4 to a human agent within 1 business hour," not "should probably follow up"

### Key Takeaway
A looping conditional edge and a one-shot fork use the *exact same* `add_conditional_edges` API — the only
difference is whether the routing function's return value points to a node that's already upstream in the
graph. The thing that actually prevents an infinite loop isn't the graph structure, it's a state-carried
exit condition (`resolved` or a turn ceiling) that the routing function checks on every pass — so that
condition is the single piece of logic in this whole notebook worth re-verifying by hand before trusting it
in front of a real customer.

### Interview framing
*"How would you explain today's work to a client or interviewer?"*
"I built a support-chat agent as a LangGraph `StateGraph` with a genuine conversational loop, not just a
single request/response. The agent and a 'customer turn' node hand control back and forth until either the
model signals resolution via an explicit tag — deliberately not a fuzzy sentiment guess, because that has to
be auditable — or a turn ceiling forces escalation to a human. Because I used a `MemorySaver` checkpointer
keyed by `thread_id`, this survives across separate calls into the graph, which matters because in production
a customer's follow-up message doesn't arrive in the same process invocation as their first one. I can also
pull the full checkpoint history for any thread, which is what a support lead would use to audit exactly what
the bot said and when it decided a case was closed."

### GitHub note
This notebook is a candidate building block for the Week 4 capstone (Days 212–216): the multi-turn +
checkpointing pattern here is the conversational core the final sellable support-chatbot repo will wrap with
an MCP tool call and a fine-tuned classifier.
